# Vorgehen

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation




# 1. Business Understanding

Grundlagen des Datensatzes:
- Synthetischer Datensatz
- 50.000 Essens-Bestellungen
- 6 indische Städte, Beträge in INR
- Merkmale
    - Bestellkontext: Uhrzeit, Wochenende/Werktag, Regen
    - Bestellung: Küche, Mahlzeit, Restauranttyp, Bestellwert, Rabatt, Liefergebühr
    - Verhalten: Bewertung, Zeit Wiederholungsbestellung
    - Situation: Stimmung, Hunger, Begleichtung
    - Alter

Background:
- Bisher werden Rabatte vermutlich an jeden gegeben. Im Datensatz bekommt ungefähr jede zweite Bestellung einen Rabatt.

Anwendungsfall:
- Vorhersage von Wiederholungsbestellungen für gezielte Rabattvergabe (Auf aktuell 50.000 Bestellungen 4.000 Nutzer)
    - Zielvariable: is_repeat_order
    - Nach jeder Bestellung schätzt das Modell, wie wahrscheinlich der Kunde wieder bestellt, je nachdem wird ein Rabatt vergeben
        - Hohe Wahrscheinlichkeit -> kein Rabatt
        - Geringe Wahrscheinlichkeit -> Rabatt


# 2. Data Understanding

Datensatz:
- https://www.kaggle.com/datasets/rhythmghai/food-ordering-behavior-india-50k-orders?resource=download
- Synthetische Daten
- Autor: Rhythm_Ghai

In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

sns.set_theme(style="whitegrid")

In [2]:
df = pd.read_csv("food_ordering_behavior_dataset.csv")
df.head()

,order_id,user_id,age,city,order_time,day_type,cuisine,meal_type,restaurant_type,order_value,discount_applied,delivery_fee,time_taken_to_order,rating_given,is_repeat_order,mood,hunger_level,company,rainy_weather
0,1,2698,35,Pune,Evening,Weekend,Chinese,Dinner,Premium,971,Yes,90,13,1,Yes,Celebrating,High,Partner,No
1,2,3237,44,Mumbai,Night,Weekend,South Indian,Dinner,Budget,442,No,26,13,2,No,Lazy,Low,Family,No
2,3,3626,31,Delhi,Morning,Weekend,Biryani,Breakfast,Mid-range,739,Yes,85,10,2,Yes,Happy,Medium,Friends,No
3,4,3176,23,Delhi,Evening,Weekend,Biryani,Snacks,Mid-range,466,No,44,12,2,No,Happy,Medium,Alone,No
4,5,4824,26,Chandigarh,Morning,Weekday,Chinese,Lunch,Premium,927,Yes,58,13,2,No,Happy,Medium,Partner,Yes


In [ ]:
# Anzahl Spalten, Zeilen
print(df.shape)

# Anzahl einzigsartiger Nutzer
print(df["user_id"].nunique())

# Spalten
df.info()

In [ ]:
num_cols = ["age", "order_value", "delivery_fee", "time_taken_to_order", "rating_given"]
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

In [ ]:
# Mittelwert, Standartabweichung, Min, Max für numerische Spalten

df[num_cols].describe().T

# Alle Ausprägung der kategorialen Spalten
for c in cat_cols:
    print(c, df[c].unique())

In [ ]:
# Zielvariabel anschauen

print(df["is_repeat_order"].value_counts(normalize=True))

# Grafisch darstellen

sns.countplot(data=df, x="is_repeat_order")
plt.show()

# 3. Data Preperation

Da es sich um synthetisch generierte und bereits qualitätsgesicherte Daten handelt, sind fehlende Werte und Duplikate erwartungsgemäß nicht vorhanden. Die Prüfung bestätigt dies. Eine weiterführende Untersuchung der Datenqualität ist an dieser Stelle daher nicht notwendig.

In [3]:
print("Fehlende Werte pro Spalte:")
print(df.isna().sum())
print("Duplizierte Zeilen:", df.duplicated().sum())
print("Duplizierte order_id:", df["order_id"].duplicated().sum())

Fehlende Werte pro Spalte:
order_id               0
user_id                0
age                    0
city                   0
order_time             0
day_type               0
cuisine                0
meal_type              0
restaurant_type        0
order_value            0
discount_applied       0
delivery_fee           0
time_taken_to_order    0
rating_given           0
is_repeat_order        0
mood                   0
hunger_level           0
company                0
rainy_weather          0
dtype: int64
Duplizierte Zeilen: 0
Duplizierte order_id: 0


In [ ]:
# Select Data
data = df.copy()

# Order-Id droppen, da es irrelevant ist
data = data.drop(columns=["order_id])

# Construct Data
# Zeile für die gesamten Kosten hinzufügen
data["total_cost"] = data["order_value"] + ["delivery_fee"]

# Anteil der Liefergebühr am Bestellwert -> hohe Gebühr könnte abschrecken
data["fee_share"] = data["delivery_fee"] / data["order_value"]

# # Altersgruppen statt einzelner Zahlen
data["age_group"] = pd.cut(data["age"], bins=[17, 24, 34, 44], labels=["18-24", "25-34", "35-44"]

# Reformatted Data
# Ja/Nein Spalten in 0/1 umwandeln

yesNoColumns = ["discount_applied", "is_repeated_order", "rainy_weather"]

for c in yesNoColumns:
    data[c] = (data[c] == "Yes").astype(int)

# Ebenfalls für day_type möglich, entweder wochenede oder unter der Woche
data["is_weekend"] = (data["day_type"] == "Weekend").astype(int)
data = data.drop(columns=["day_type"])